# Course 8: Mathematics and AI
## Session 3: Steerable kernels

**Goal:** solve the kernel constraint numerically, look at the basis of allowed kernels, and build an equivariant convolution from it.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. The constraint
A convolution maps a feature field of type `rho_in` to one of type `rho_out`. It is equivariant to a rotation group G exactly when its kernel satisfies

`k(g x) = rho_out(g) k(x) rho_in(g)^-1` for every g in G and every position x.

This is linear in k, so the allowed kernels form a vector space. We find a basis with an SVD. To keep everything exact on a pixel grid, G is C4 (rotations by 90 degrees). The same construction with continuous angles gives SO(2) and the circular harmonics.

In [ ]:
import numpy as np
K = 5                                                             # kernel size
coords = [(i, j) for i in range(K) for j in range(K)]
def rot_index(i, j):                                              # where pixel (i, j) goes under a quarter turn of the grid
    return (j, K - 1 - i)                                         # chosen to match torch.rot90(x, 1), tested in section 3
perm = np.zeros((K * K, K * K))
for a, (i, j) in enumerate(coords): perm[coords.index(rot_index(i, j)), a] = 1

R = np.array([[0.0, -1.0], [1.0, 0.0]])
REPS = {"trivial": np.eye(1), "sign": -np.eye(1), "vector": R, "regular": np.roll(np.eye(4), 1, axis=0)}      # the generator's matrix in each representation

def kernel_basis(rep_in, rep_out):
    """Basis of kernels k with k(g x) = rho_out(g) k(x) rho_in(g)^-1, for the generator g (a quarter turn)."""
    A, B = REPS[rep_out], REPS[rep_in]; co, ci = len(A), len(B)
    # unknown: k as a vector of size K*K*co*ci. Constraint: (perm ⊗ I) k - (I ⊗ (A ⊗ B^-T)) k = 0
    M = np.kron(perm, np.eye(co * ci)) - np.kron(np.eye(K * K), np.kron(A, np.linalg.inv(B).T))
    _, s, Vt = np.linalg.svd(M)
    return Vt[np.sum(s > 1e-9):].reshape(-1, K, K, co, ci)

for pair in [("trivial", "trivial"), ("trivial", "sign"), ("trivial", "vector"), ("vector", "vector"), ("regular", "regular")]:
    print(f"{pair[0]:8s} -> {pair[1]:8s}: {len(kernel_basis(*pair)):3d} basis kernels  (an unconstrained kernel would have {K * K * len(REPS[pair[0]]) * len(REPS[pair[1]])} free numbers)")
assert len(kernel_basis("trivial", "trivial")) == 7                 # one per orbit of pixels: the centre plus 24 / 4

Each count has a meaning. For scalar to scalar, the kernel must look the same after a quarter turn, so there is one free number per orbit of pixels: 7. The constraint removes about three quarters of the parameters, and the ones that remain are exactly those that cannot break the symmetry.

### 2. Look at the basis

In [ ]:
import matplotlib.pyplot as plt
basis_ss = kernel_basis("trivial", "trivial"); basis_sv = kernel_basis("trivial", "vector")
fig, axes = plt.subplots(2, 7, figsize=(14, 4.5))
for a in range(7):
    axes[0, a].imshow(basis_ss[a, :, :, 0, 0], cmap="RdBu"); axes[0, a].set_title(f"scalar->scalar {a}", fontsize=8)
    k = basis_sv[a % len(basis_sv)]; yy, xx = np.mgrid[0:K, 0:K]
    axes[1, a].quiver(xx, yy, k[:, :, 0, 0], -k[:, :, 1, 0]); axes[1, a].invert_yaxis(); axes[1, a].set_title(f"scalar->vector {a % len(basis_sv)}", fontsize=8)
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
fig.savefig("kernel_basis.png")
k = basis_ss[0, :, :, 0, 0]; print("a scalar->scalar basis kernel equals its own quarter turn:", np.allclose(k, np.rot90(k)))
print("scalar->vector kernels at the centre pixel are all zero:", np.allclose(basis_sv[:, 2, 2], 0), "(a vector that must equal its own rotation is zero)")
assert np.allclose(k, np.rot90(k)) and np.allclose(basis_sv[:, 2, 2], 0)

The scalar-to-vector kernels are little vector fields that turn with the grid: swirls and sources. An edge detector that outputs a gradient direction is built from these.

### 3. Build a layer and test it
A steerable convolution is a learned mix of the basis kernels. Rotate the input image, and the output field should rotate too, with its vectors turned by the same angle.

In [ ]:
import torch, torch.nn.functional as F
torch.manual_seed(0)
w = torch.randn(len(basis_sv), dtype=torch.float64)
kernel = torch.einsum("b,bijoc->ocij", w, torch.tensor(basis_sv))            # (out channels = 2, in channels = 1, K, K)
conv = lambda x: F.conv2d(x, kernel, padding=K // 2)
x = torch.randn(1, 1, 12, 12, dtype=torch.float64)

out_of_rotated = conv(torch.rot90(x, 1, (2, 3)))                             # rotate the image, then convolve
expected = torch.einsum("ab,nbij->naij", torch.tensor(R), torch.rot90(conv(x), 1, (2, 3)))   # convolve, rotate the field, turn each vector
err = (out_of_rotated - expected).abs().max().item()
plain = torch.randn(2, 1, K, K, dtype=torch.float64)
err_plain = (F.conv2d(torch.rot90(x, 1, (2, 3)), plain, padding=2) - torch.einsum("ab,nbij->naij", torch.tensor(R), torch.rot90(F.conv2d(x, plain, padding=2), 1, (2, 3)))).abs().max().item()
print(f"steerable kernel: equivariance error {err:.2e} | unconstrained kernel: {err_plain:.2f}")
assert err < 1e-9 and err_plain > 0.1

If the test fails with the vectors turning the wrong way, the sign convention of your grid rotation and of `R` disagree. That mismatch is the most common bug in hand-written equivariant code, and this test is how you find it.

### 4. From C4 to SO(2): circular harmonics
For continuous rotations the solutions separate into a free radial profile times a fixed angular part `exp(i m phi)`. A kernel from a frequency-m field to a frequency-n field may only use angular frequencies `n - m` and `n + m`. Check the simplest case on sampled points: a kernel `k(r, phi) = f(r) * (cos phi, sin phi)` maps scalars to vectors.

In [ ]:
phi = np.linspace(0, 2 * np.pi, 360, endpoint=False); theta = 0.7
rot = lambda t: np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
k = lambda p: np.stack([np.cos(p), np.sin(p)])                    # angular part of a scalar->vector kernel (frequency 1)
lhs = k(phi + theta); rhs = rot(theta) @ k(phi)
print("k(g x) = rho_out(g) k(x) for a rotation by 0.7 rad:", np.allclose(lhs, rhs))
k2 = lambda p: np.stack([np.cos(2 * p), np.sin(2 * p)])           # frequency 2: wrong for a frequency-1 output
print("frequency-2 angular part satisfies it:", np.allclose(k2(phi + theta), rot(theta) @ k2(phi)))
assert np.allclose(lhs, rhs)

### TODO
1. Add the reflection to get D4: stack a second constraint for the flip into `M` and recompute the basis sizes.
2. For regular -> regular, show that the solutions are exactly the group convolution kernels of Cohen and Welling (one free 5x5 kernel, copied in four rotations).
3. Count parameters: a 5x5 layer with 16 regular fields in and out, constrained and unconstrained.